# Aprendizagem não supervisionada

A aprendizagem não supervisionada é uma área da aprendizagem automática que se foca na extração de padrões e estruturas interessantes de dados não etiquetados. Em Python, a biblioteca [scikit-learn](https://scikit-learn.org/) é uma das ferramentas mais utilizadas para aplicar abordagens de aprendizagem não supervisionada e de aprendizagem automática em geral. Com uma vasta coleção de algoritmos de agrupamento (*clustering*), redução de dimensionalidade e deteção de anomalias, a biblioteca *scikit-learn* oferece uma ampla gama de funcionalidades para explorar e compreender conjuntos de dados não etiquetados. Este tutorial foca-se maioritariamente na aplicação de abordagens de *clustering*, começando pelo algoritmo *k-Means*. Para obter informação sobre outras funcionalidades disponibilizadas pela biblioteca *scikit-learn*, recomendamos a consulta da sua [documentação](https://scikit-learn.org/stable/user_guide.html).

In [ ]:
import sklearn

In [ ]:
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_theme() 

## Manipulação de dados

Para além de implementações eficientes de algoritmos de aprendizagem automática, a biblioteca *scikit-learn* também disponibiliza algumas funcionalidades de manipulação de dados relevantes no contexto da aplicação desses algoritmos. Por exemplo, normalização e redução de dimensionalidade. Para demonstrar algumas destas funcionalidades, vamos usar o conjunto de dados [Iris](https://archive.ics.uci.edu/dataset/53/iris) que já vimos anteriormente:

In [ ]:
iris = sns.load_dataset('iris')
iris.sample(5)

**Nota**: O método `sample` escolhe aleatoriamente uma amostra de *n* exemplos do conjunto de dados.

In [ ]:
sns.pairplot(iris, hue='species');

Num contexto de aprendizagem não supervisionada, a espécie de planta serve apenas como uma indicação de uma possível separação dos dados que tem um significado conhecido no mundo real. Para além disso, mesmo que estivessemos num contexto de aprendizagem supervisionada, a biblioteca *scikit-learn* lida separadamente com os atributos usados para representar os dados e aquele ou aqueles que representam o que se pretende prever. Como tal, para simplificar a aplicação das funções da biblioteca, vamos criar uma nova tabela de dados sem o atributo `species`:

In [ ]:
iris_features = iris.drop(columns=['species'])
iris_features.head()

**Nota**: A subdivisão dos exemplos por espécie de planta vai continuar a ser usada como apoio em algumas visualizações.

### Normalização

A normalização dos valores dos atributos pode ter um impacto relevante quando estes têm escalas diferentes. A classe `StandardScaler` da biblioteca *scikit-learn* pode ser usada para fazer normalização baseada em *z-score*, isto é, transformar os atributos de forma a que a distribuição dos seus valores tenha média zero e desvio padrão unitário.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
std_scaler = StandardScaler()
std_scaler.fit(iris_features)
scaled_features = std_scaler.transform(iris_features)
std_scaled_iris = pd.DataFrame(scaled_features, index=iris_features.index, columns=iris_features.columns)
std_scaled_iris.describe()

In [ ]:
sns.pairplot(std_scaled_iris);

**Nota**: O método `fit` é usado para ajustar o normalizador, isto é, extrair dos dados a informação necessária para fazer a normalização (neste caso, a média e o desvio padrão de cada atributo), enquanto o método `transform` é usado para aplicar a normalização. O método `transform` pode ser aplicado sobre exemplos diferentes daqueles que foram usados para ajustar o normalizador. Isto permite normalizar novos dados da mesma forma que os do conjunto de dados original. O método `fit_transform` pode ser usado para efetuar as duas operações sequencialmente sobre o mesmo conjunto de dados, o que é conveniente em muitos casos.

Outra abordagem de normalização é escalar os valores dos atributos de forma a que estes fiquem dentro de um determinado intervalo, como por exemplo entre 0 e 1. Isso pode ser feito usando a funcionalidade da classe `MinMaxScaler`:

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
minmax_scaler = MinMaxScaler((0, 1))
minmax_scaled_iris = pd.DataFrame(minmax_scaler.fit_transform(iris_features), index=iris_features.index, columns=iris_features.columns)
minmax_scaled_iris.describe()

In [ ]:
sns.pairplot(minmax_scaled_iris);

### Redução de dimensionalidade

Muitas vezes é necessário reduzir a dimensionalidade de um conjunto de dados para o conseguir analisar, visualizar, ou até mesmo para melhorar o desempenho de modelos de aprendizagem automática obtidos a partir desses dados. Existem várias abordagens para reduzir a dimensionalidade, sendo as mais comuns baseadas na seleção de um subconjunto representativo dos exemplos e/ou atributos originais de acordo com um determinado critério. Outra abordagem possível é fazer uma análise de componentes principais ([PCA](https://en.wikipedia.org/wiki/Principal_component_analysis)) e usar como atributos os componentes que refletem a maior variância no conjunto de dados. Para isso, pode ser usada a classe `PCA` da biblioteca *scikit-learn*:

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=2)
pca.fit(iris_features)
pca_iris = pd.DataFrame(pca.transform(iris_features), index=iris_features.index, columns=['PC1', 'PC2'])
pca_iris.head()

**Nota**: Neste caso, escolhemos os dois componentes principais, o que nos ajuda a visualizar o conjunto de dados. Esta é uma abordagem comum, mesmo em casos em que os atributos originais são usados para fazer a restante análise do conjunto de dados.

In [ ]:
sns.scatterplot(pca_iris, x='PC1', y='PC2', hue=iris['species']);

Para verificar se os componentes selecionados são suficientemente representativos do conjunto de dados, podemos olhar para o rácio da variância total no conjunto de dados que estes explicam. Esta informação é dada pelo atributo `explained_variance_ratio_`:

In [ ]:
print(pca.explained_variance_ratio_)
print(pca.explained_variance_ratio_.sum() * 100)

Neste caso, podemos verificar que os dois componentes explicam perto de 98% da variância total no conjunto de dados.

## Agrupamento (*Clustering*)

O agrupamento, mais conhecido pelo termo inglês *clustering*, é uma técnica de aprendizagem não supervisionada que consiste em agrupar um conjunto de dados em grupos (clusters) em que os elementos de cada grupo são mais semelhantes entre si do que aos elementos de outros grupos. A biblioteca *scikit-learn* implementa vários algoritmos de *clustering*. Vamos começar por explorar o algoritmo [k-Means](#k-Means). Mais tarde, abordaremos outras abordagens, como o agrupamento hierárquico aglomerativo e o DBSCAN.

### k-Means

O algoritmo [k-Means](https://en.wikipedia.org/wiki/K-means_clustering) agrupa os exemplos de um conjunto de dados de forma a que estes fiquem separados em *k* clusters de igual variância, minimizando o valor da soma dos quadrados das distâncias de cada exemplo ao centroide do seu cluster (*within-cluster sum-of-squares*). O algoritmo requer que o número de clusters, *k*, seja específicado. Esta abordagem de *clustering* escala bem para conjuntos de dados com muitos exemplos e é a mais usada no contexto da aprendizagem não supervisionada.

Na biblioteca *scikit-learn*, o algortimo *k-Means* é implementado pela classe `KMeans`, sendo o número de clusters definido pelo argumento `n_clusters`. Neste caso, vamos começar por usar um valor de *k=3* para comparação com a classificação das plantas por espécie:

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
kmeans = KMeans(n_clusters=3, n_init='auto').fit(iris_features) # The n_init='auto' is only explicited to avoid a warning

**Nota**: A implementação do algoritmo k-Means na biblioteca *scikit-learn* inclui algumas otimizações. Por exemplo, a forma predefinida para escolha dos centroides iniciais segue o algoritmo [k-means++](https://en.wikipedia.org/wiki/K-means%2B%2B). Para além disso, esta implementação só permite o uso da distância euclidiana como função de distância.

Os centroides dos clusters obtidos e as atribuições dos exemplos aos clusters são dados pelos atributos `cluster_centers_` e `labels_`, respetivamente: 

In [ ]:
kmeans.cluster_centers_

In [ ]:
kmeans.labels_

Vamos guardar este agrupamento na tabela de dados para podermos visualizá-lo e, mais tarde, compará-lo com os agrupamentos obtidos usando outros algoritmos: 

In [ ]:
iris['kmeans'] = pd.Categorical(kmeans.labels_)

In [ ]:
sns.pairplot(iris, hue='kmeans');

Também podemos visualizar o agrupamento usando a projeção dos dados nos dois componentes principais obtidos através da aplicação de [PCA](#Redução-de-dimensionalidade):

In [ ]:
sns.scatterplot(pca_iris, x='PC1', y='PC2', hue=iris['kmeans']);

Podemos ver que o agrupamento obtido é distinto da classificação das plantas por espécie, mas tem algumas semelhanças:

In [ ]:
sns.scatterplot(pca_iris, x='PC1', y='PC2', hue=iris['species']);

O atributo `inertia_` guarda o valor da soma dos quadrados das distâncias de cada exemplo ao centroide do seu cluster (*within-cluster sum of squares*):

In [ ]:
kmeans.inertia_

Podemos usar esta informação para analisar a qualidade do agrupamento para vários valores de *k*:

In [ ]:
k_wss = pd.DataFrame([(k, KMeans(n_clusters=k, n_init='auto').fit(iris_features).inertia_) for k in range(1, 11)], columns=['k', 'wss'])
sns.lineplot(k_wss, x='k', y='wss');

Usando o método do "cotovelo" podemos concluir que *k=3* é um valor adequado para o número de clusters.

**Nota**: Na próxima aula vamos abordar outras abordagens de *clustering* e outras formas de avaliar a qualidade dos clusters obtidos.